# Modelo 1 — Bayesian MCMC (PyMC): Francia vs Paraguay (versión LOCAL / Jupyter)

Esta es la versión adaptada para correr **localmente con Anaconda + Jupyter Notebook/Lab**,
en lugar de Google Colab. Mantiene exactamente la misma lógica de negocio que la versión
de Colab (`notebooks/Modelo1_Bayesian_MCMC_Francia_Paraguay.ipynb`):

- Mismo dataset: [martj42/international_results](https://github.com/martj42/international_results)
- Mismo target: `0` = gana local, `1` = empate, `2` = gana visitante
- Mismo feature engineering (>100 variables, sin data leakage)
- Mismo split temporal 80/20
- **Todas** las variables generadas se usan en el modelo Bayesiano (no se recorta a un
  top-K de variables como en la versión de Colab; en su lugar se usa un *prior
  jerárquico de regularización* — ver Sección 8 — para poder manejar 180 variables sin
  descartar ninguna).

Diferencias principales frente a Colab: sin `google.colab`, sin rutas `/content/`, con
variables `REPO_PATH` / `DATA_PATH` configurables, guardado del modelo en disco local, y
utilidades para no re-entrenar en cada sesión. El detalle completo de los cambios está en
la última sección de este notebook.


## 0. Configuración para correr localmente

### 0.1 Crear el entorno con Anaconda

Abre **Anaconda Prompt** (en Windows) o una terminal con `conda` disponible (Mac/Linux)
y ejecuta, uno por uno:

```bash
conda create -n futbol-bayes python=3.11 -y
conda activate futbol-bayes
pip install pandas numpy scipy scikit-learn pymc arviz matplotlib seaborn xgboost jupyterlab ipykernel
python -m ipykernel install --user --name futbol-bayes --display-name "Python (futbol-bayes)"
jupyter lab
```

- La línea `conda create` crea un entorno aislado llamado `futbol-bayes` con Python 3.11
  (versión estable y compatible con PyMC 5 al momento de escribir esto).
- `conda activate futbol-bayes` activa ese entorno en la terminal actual.
- `pip install ...` instala todas las librerías necesarias dentro del entorno.
- `python -m ipykernel install ...` registra el entorno como un **kernel de Jupyter**
  seleccionable, con el nombre visible `"Python (futbol-bayes)"`.
- `jupyter lab` abre Jupyter Lab en el navegador.

### 0.2 Abrir este notebook y seleccionar el kernel correcto

1. En Jupyter Lab, abre este archivo `.ipynb` (File → Open, o arrastrándolo a la ventana).
2. En la esquina superior derecha del notebook verás el nombre del kernel activo.
   Haz clic ahí (o ve a **Kernel → Change Kernel...**).
3. Selecciona **"Python (futbol-bayes)"** de la lista.
4. Verifica que quedó bien ejecutando la siguiente celda de imports: si no hay errores
   de `ModuleNotFoundError`, el kernel está correctamente configurado.

### 0.3 Notas para Windows

- Usa siempre `r"..."` (raw strings) para rutas de Windows con backslashes, por ejemplo
  `r"C:\Users\MI_USUARIO\Documents\international_results"`, para evitar que Python
  interprete `\U`, `\D`, etc. como caracteres de escape.
- Este notebook usa `pathlib.Path` en vez de concatenar strings de rutas, lo cual
  funciona igual en Windows, Mac y Linux.


## 1. Importación de librerías

Ya no se usa `!pip install` dentro del notebook (eso es un "magic command" pensado para
entornos efímeros como Colab). Las librerías se instalan **una sola vez** al crear el
entorno de conda (Sección 0). Si falta alguna, la celda de abajo te dirá cuál.

In [ ]:
# ============================================================
# 1. IMPORTS (version local - sin comandos exclusivos de Colab)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os
import subprocess
import time
import pickle
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pymc as pm
import arviz as az
import pytensor.tensor as pt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("PyMC version:", pm.__version__)
print("ArviZ version:", az.__version__)
print("Directorio de trabajo actual:", Path.cwd())


## 2. Configuración de rutas locales: `REPO_PATH` y `DATA_PATH`

Reemplaza las rutas `/content/...` de Colab por rutas locales configurables:

- **`REPO_PATH`**: carpeta donde está (o donde quieres clonar) el repositorio de datos.
  Puede ser una ruta relativa (`"international_results"`, relativa a donde abriste
  Jupyter) o absoluta, por ejemplo en Windows:
  `r"C:\Users\MI_USUARIO\Documents\international_results"`.
- **`DATA_PATH`**: si ya tienes el CSV descargado y no quieres depender de `git`, apunta
  directamente al archivo (por ejemplo
  `r"C:\Users\MI_USUARIO\Documents\international_results\results.csv"`). Si lo dejas en
  `None`, el notebook busca el CSV automáticamente dentro de `REPO_PATH`.

**Opción A (recomendada si tienes `git` instalado):** deja `DATA_PATH = None` y el
notebook clona el repo con `git clone` dentro de `REPO_PATH` si todavía no existe ahí.

**Opción B (sin git):** descarga manualmente el ZIP del repo desde GitHub, descomprímelo,
y pon la ruta a la carpeta (o directo al CSV) en `REPO_PATH` / `DATA_PATH`.

In [ ]:
# ============================================================
# 2. CONFIGURACION DE RUTAS (reemplaza /content/... de Colab)
# ============================================================
REPO_URL = "https://github.com/martj42/international_results"

# --- CONFIGURA ESTAS DOS VARIABLES SEGUN TU COMPUTADORA ---

# Carpeta local donde esta (o donde se clonara) el repositorio.
# Puede ser relativa a la carpeta donde abriste Jupyter, o absoluta.
REPO_PATH = "international_results"
# Ejemplo en Windows con ruta absoluta:
# REPO_PATH = r"C:\Users\MI_USUARIO\Documents\international_results"

# Si ya tienes el CSV en una ubicacion especifica, ponla aqui y se usara
# directamente (sin buscar ni clonar nada). Si no, deja DATA_PATH = None.
DATA_PATH = None
# Ejemplo en Windows:
# DATA_PATH = r"C:\Users\MI_USUARIO\Documents\international_results\results.csv"

# ------------------------------------------------------------------

repo_path = Path(REPO_PATH).expanduser().resolve()


def find_results_csv(search_dirs):
    """
    Busca de forma flexible un archivo CSV de resultados dentro de una lista de
    directorios locales. Prioriza nombres tipicos ('results.csv') y si no los
    encuentra, busca cualquier CSV que contenga las columnas esperadas.
    Funciona igual en Windows, Mac y Linux gracias a pathlib.
    """
    candidate_names = ["results.csv", "result.csv", "matches.csv", "international_results.csv"]
    csv_files = []
    for base_dir in search_dirs:
        base_dir = Path(base_dir)
        if not base_dir.is_dir():
            continue
        csv_files.extend(base_dir.rglob("*.csv"))

    # 1. Coincidencia exacta por nombre
    for name in candidate_names:
        for path in csv_files:
            if path.name.lower() == name:
                return path

    # 2. Si no hay coincidencia por nombre, buscar por columnas esperadas
    expected_cols = {"home_team", "away_team", "home_score", "away_score"}
    for path in csv_files:
        try:
            sample = pd.read_csv(path, nrows=5)
            if expected_cols.issubset(set(c.lower() for c in sample.columns)):
                return path
        except Exception:
            continue

    return None


if DATA_PATH is not None:
    csv_path = Path(DATA_PATH).expanduser().resolve()
    if not csv_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo indicado en DATA_PATH: {csv_path}")
else:
    csv_path = find_results_csv([repo_path, Path.cwd()])

    if csv_path is None:
        print(f"No se encontro un CSV valido en {repo_path}. Intentando clonar el repositorio...")
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_path)], check=True)
            csv_path = find_results_csv([repo_path])
        except Exception as e:
            print("No se pudo clonar automaticamente con git. Puedes:")
            print(f"  a) Instalar git y volver a correr esta celda, o")
            print(f"  b) Descargar el ZIP manualmente desde {REPO_URL} y descomprimirlo en:")
            print(f"     {repo_path}")
            print("Detalle del error:", e)

    if csv_path is None:
        raise FileNotFoundError(
            "No se pudo localizar results.csv. Revisa REPO_PATH/DATA_PATH o descarga "
            "el repo manualmente (ver instrucciones arriba)."
        )

print("Archivo de resultados encontrado en:", csv_path)

raw_df = pd.read_csv(csv_path)
raw_df.columns = [c.strip().lower() for c in raw_df.columns]
print("Shape original:", raw_df.shape)
raw_df.head()


## 3. Limpieza del dataset

Idéntico a la versión de Colab: normalizamos columnas, convertimos `date`, ordenamos
cronológicamente, quitamos partidos sin marcador y creamos el `target`.

In [ ]:
# ============================================================
# 3. LIMPIEZA DEL DATASET
# ============================================================
df = raw_df.copy()

# Normalizar nombres de columnas por si vienen con variantes
rename_map = {
    "hometeam": "home_team", "awayteam": "away_team",
    "homescore": "home_score", "awayscore": "away_score",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el dataset: {missing}")

# Convertir fecha y descartar filas sin fecha valida
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Ordenar cronologicamente (necesario para que el feature engineering no tenga leakage)
df = df.sort_values("date").reset_index(drop=True)

# Eliminar partidos sin marcador valido
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Columnas opcionales manejadas de forma flexible
if "tournament" not in df.columns:
    df["tournament"] = "Unknown"

if "neutral" not in df.columns:
    df["neutral"] = False
else:
    df["neutral"] = (
        df["neutral"].astype(str).str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)
    )
df["neutral"] = df["neutral"].astype(int)

# id unico de partido (se usa para reconstruir las features mas adelante)
df["match_id"] = np.arange(len(df))

# Variable objetivo (target)
# 0 = gana home_team | 1 = empate | 2 = gana away_team
conditions = [
    df["home_score"] > df["away_score"],
    df["home_score"] == df["away_score"],
    df["home_score"] < df["away_score"],
]
df["target"] = np.select(conditions, [0, 1, 2]).astype(int)

print(f"Dataset limpio: {df.shape[0]} partidos, {df.shape[1]} columnas")
print(df["target"].value_counts(normalize=True).rename("proporcion"))
df[["date", "home_team", "away_team", "home_score", "away_score", "target"]].head()


## 4. Feature engineering (>100 variables, sin data leakage)

Misma lógica exacta que en Colab: formato largo por equipo, `shift(1)` antes del
`rolling` para que el partido actual nunca influya en sus propias variables, ventanas
de 3/5/10/15/20 partidos, 12 métricas por ventana → **180 variables** (`home_*`,
`away_*`, `diff_*`). No se recorta nada aquí.

In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING (SIN DATA LEAKAGE) - SIN CAMBIOS DE LOGICA
# ============================================================
WINDOWS = [3, 5, 10, 15, 20]

# --- 4.1 Formato largo: una fila por equipo y partido ---
long_cols = ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]

home_long = df[long_cols].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "goals_for", "away_score": "goals_against",
})
home_long["is_home"] = 1

away_long = df[long_cols].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "goals_for", "home_score": "goals_against",
})
away_long["is_home"] = 0

long_df = pd.concat([home_long, away_long], ignore_index=True)
long_df = long_df.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

# --- 4.2 Metricas base por partido (antes de aplicar ventanas) ---
long_df["goal_diff"] = long_df["goals_for"] - long_df["goals_against"]
long_df["win"] = (long_df["goals_for"] > long_df["goals_against"]).astype(int)
long_df["draw"] = (long_df["goals_for"] == long_df["goals_against"]).astype(int)
long_df["loss"] = (long_df["goals_for"] < long_df["goals_against"]).astype(int)
long_df["points"] = long_df["win"] * 3 + long_df["draw"] * 1
long_df["clean_sheet"] = (long_df["goals_against"] == 0).astype(int)
long_df["failed_to_score"] = (long_df["goals_for"] == 0).astype(int)

# 9 metricas "mean/rate" + 3 metricas de desviacion estandar = 12 metricas por ventana
BASE_METRICS = ["goals_for", "goals_against", "goal_diff", "points",
                "win", "draw", "loss", "clean_sheet", "failed_to_score"]
STD_METRICS = ["goals_for", "goals_against", "goal_diff"]

# --- 4.3 Rolling con shift(1): usa SOLO partidos anteriores al actual ---
feature_cols_long = []
grouped = long_df.groupby("team", group_keys=False)

t0 = time.time()
new_cols = {}
for w in WINDOWS:
    for col in BASE_METRICS:
        feat_name = f"{col}_mean_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        feature_cols_long.append(feat_name)
    for col in STD_METRICS:
        feat_name = f"{col}_std_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).std())
        feature_cols_long.append(feat_name)

long_df = pd.concat([long_df, pd.DataFrame(new_cols, index=long_df.index)], axis=1)
print(f"Variables de forma por equipo calculadas en {time.time() - t0:.1f}s -> {len(feature_cols_long)} columnas")

# --- 4.4 Volver a formato ancho: features del local, del visitante y diferencias ---
rename_home = {c: f"home_{c}" for c in feature_cols_long}
rename_away = {c: f"away_{c}" for c in feature_cols_long}

home_features = long_df.loc[long_df.is_home == 1, ["match_id"] + feature_cols_long].rename(columns=rename_home)
away_features = long_df.loc[long_df.is_home == 0, ["match_id"] + feature_cols_long].rename(columns=rename_away)

df = df.merge(home_features, on="match_id", how="left").merge(away_features, on="match_id", how="left")

diff_data = {f"diff_{c}": df[f"home_{c}"] - df[f"away_{c}"] for c in feature_cols_long}
df = pd.concat([df, pd.DataFrame(diff_data, index=df.index)], axis=1)
diff_cols = list(diff_data.keys())

# all_feature_cols = TODAS las variables generadas (no se recorta ninguna)
all_feature_cols = [f"home_{c}" for c in feature_cols_long] + [f"away_{c}" for c in feature_cols_long] + diff_cols
print(f"Total de variables generadas: {len(all_feature_cols)}")

# Partidos al inicio del historico de un equipo no tienen ventana previa -> NaN.
# Se rellenan con 0 (equivalente a "sin informacion previa disponible").
df[all_feature_cols] = df[all_feature_cols].fillna(0)

assert len(all_feature_cols) > 100, "Se esperaban mas de 100 variables"
df[all_feature_cols].describe().T.head()


## 5. Separación temporal train / test (80% / 20%)

Sin cambios: los primeros 80% de partidos (cronológicamente) para entrenar, el último
20% para test. Nada de `train_test_split` aleatorio.

In [ ]:
# ============================================================
# 5. SEPARACION TEMPORAL TRAIN / TEST (80% / 20%)
# ============================================================
df_model = df.sort_values("date").reset_index(drop=True)

split_idx = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_idx].copy()
test_df = df_model.iloc[split_idx:].copy()

print(f"Train: {train_df.shape[0]} partidos "
      f"({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"Test:  {test_df.shape[0]} partidos "
      f"({test_df['date'].min().date()} a {test_df['date'].max().date()})")

y_train = train_df["target"].values.astype(int)
y_test = test_df["target"].values.astype(int)


## 6. Escalado de variables (se usan las 180 variables, sin recorte)

A diferencia de la versión de Colab (que reducía a un top-40 por importancia para
acelerar el muestreo dentro del límite de tiempo de Colab), aquí **se conservan todas
las variables generadas**, tal como lo pediste. El costo de esto se paga en tiempo de
cómputo del MCMC (Sección 8), lo cual es aceptable corriendo localmente sin límite de
sesión.

Para que el modelo siga siendo manejable con 180 predictores (muchos de ellos
correlacionados entre sí, por ejemplo las medias de ventanas de 15 y 20 partidos), en
la Sección 8 se usa un **prior jerárquico de regularización (shrinkage)** en lugar de
descartar variables: cada grupo de coeficientes comparte una escala común que el propio
modelo aprende de los datos, encogiendo hacia cero los coeficientes poco informativos
sin eliminar ninguna variable del modelo.

In [ ]:
# ============================================================
# 6. ESCALADO - TODAS LAS VARIABLES (sin seleccion de top-K)
# ============================================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(train_df[all_feature_cols].values)
X_test_scaled = scaler.transform(test_df[all_feature_cols].values)

print("Shape final para PyMC -> train:", X_train_scaled.shape, "| test:", X_test_scaled.shape)
print(f"Numero de variables usadas por el modelo: {len(all_feature_cols)} (todas)")


## 7. Evitar que Windows suspenda la computadora durante el entrenamiento

Con 180 variables el muestreo MCMC puede tardar bastante más que con 40 (varios
minutos a más de una hora, según tu CPU). Si Windows suspende el equipo a la mitad,
el kernel de Jupyter se congela y pierdes el avance.

**Opción recomendada (config. de Windows, sin código):**
1. Ve a **Configuración → Sistema → Energía y batería → Pantalla y suspensión**.
2. Pon "Cuando esté conectado, poner el dispositivo en suspensión después de" en
   **Nunca** mientras dure el entrenamiento (puedes revertirlo después).
3. Alternativa por línea de comandos (Símbolo del sistema como administrador):
   ```
   powercfg /change standby-timeout-ac 0
   ```
   Para revertirlo luego (ej. a 30 minutos):
   ```
   powercfg /change standby-timeout-ac 30
   ```

**Opción alternativa (código, solo Windows):** la celda siguiente usa la API de Windows
(`SetThreadExecutionState`) para pedirle al sistema que no suspenda el equipo mientras
el kernel de Jupyter esté vivo. Es un complemento, no un reemplazo, de la configuración
de energía: úsala junto con el paso 1/2 de arriba.

In [ ]:
# ============================================================
# 7. MANTENER LA PC ACTIVA DURANTE EL ENTRENAMIENTO (solo Windows)
# ============================================================
import ctypes

ES_CONTINUOUS = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ES_AWAYMODE_REQUIRED = 0x00000040


def prevent_sleep():
    """Le pide a Windows que no suspenda el equipo mientras el kernel siga activo."""
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(
            ES_CONTINUOUS | ES_SYSTEM_REQUIRED | ES_AWAYMODE_REQUIRED
        )
        print("OK: se solicito a Windows no suspender el equipo mientras dure el entrenamiento.")
    except AttributeError:
        print("Esta funcion es especifica de Windows. En Mac/Linux, ajusta la configuracion "
              "de energia del sistema operativo (ej. 'caffeinate' en Mac, o los ajustes de "
              "suspension en la configuracion de energia de Linux).")


def allow_sleep():
    """Restaura el comportamiento normal de suspension de Windows."""
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS)
        print("OK: se restauro el comportamiento normal de suspension de Windows.")
    except AttributeError:
        pass


## 8. Modelo Bayesiano multinomial en PyMC (MCMC/NUTS) — con todas las variables

Misma estructura que en Colab: regresión logística multinomial con clase base
`home_team gana` (`eta_home = 0`), y dos "logits" modelados explícitamente:

- `eta_draw = intercept_draw + X · beta_draw`
- `eta_away = intercept_away + X · beta_away`

La única diferencia respecto a Colab es el prior de `beta_draw` / `beta_away`: en vez
de un `Normal(0, 1)` fijo, cada uno usa un **prior jerárquico** `Normal(0, sigma_beta)`
donde `sigma_beta ~ HalfNormal(1)` es una escala compartida que el modelo aprende. Con
180 variables (muchas correlacionadas), esto regulariza los coeficientes de forma
automática — sin necesidad de descartar ninguna variable, tal como pediste. El resto
del modelo (softmax, `pm.Categorical`, configuración de `pm.sample`) es idéntico a la
versión de Colab.

El modelo se define dentro de una función `build_bayesian_model(...)` para poder
**reconstruir la misma arquitectura más adelante** (Sección 11) sin tener que reentrenar,
cuando cargues el modelo guardado en una sesión nueva.

In [ ]:
# ============================================================
# 8. MODELO BAYESIANO MULTINOMIAL EN PyMC (TODAS LAS VARIABLES)
# ============================================================
def build_bayesian_model(X, y, feature_names):
    """
    Construye el modelo de regresion logistica multinomial Bayesiana.
    Clase base = gana home_team (eta_home = 0 fijo).
    Se modelan eta_draw y eta_away con priors jerarquicos que regularizan
    los 180 coeficientes sin necesidad de descartar variables.
    """
    coords = {"feature": feature_names}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data("X_data", X)   # (n_obs, n_features), mutable
        y_data = pm.Data("y_data", y)   # (n_obs,), mutable

        intercept_draw = pm.Normal("intercept_draw", mu=0, sigma=2.5)
        intercept_away = pm.Normal("intercept_away", mu=0, sigma=2.5)

        # Prior jerarquico (shrinkage): con >100 variables correlacionadas entre si,
        # cada beta comparte una escala comun que el modelo aprende de los datos,
        # regularizando automaticamente sin eliminar ninguna variable.
        sigma_beta_draw = pm.HalfNormal("sigma_beta_draw", sigma=1.0)
        sigma_beta_away = pm.HalfNormal("sigma_beta_away", sigma=1.0)

        beta_draw = pm.Normal("beta_draw", mu=0, sigma=sigma_beta_draw, dims="feature")
        beta_away = pm.Normal("beta_away", mu=0, sigma=sigma_beta_away, dims="feature")

        eta_draw = intercept_draw + pm.math.dot(X_data, beta_draw)
        eta_away = intercept_away + pm.math.dot(X_data, beta_away)
        eta_home = pt.zeros_like(eta_draw)  # clase base fija en 0

        eta = pt.stack([eta_home, eta_draw, eta_away], axis=1)   # (n_obs, 3)
        p = pm.Deterministic("p", pm.math.softmax(eta, axis=1))  # probabilidades por clase

        y_obs = pm.Categorical("y_obs", p=p, observed=y_data)
    return model


bayes_model = build_bayesian_model(X_train_scaled, y_train, all_feature_cols)
print(f"Modelo construido con {len(all_feature_cols)} variables (todas las generadas).")


In [ ]:
# ============================================================
# ENTRENAMIENTO (MCMC / NUTS) - puede tardar varios minutos con 180 variables
# ============================================================
# En Windows + Jupyter, si el muestreo se queda "colgado" al usar mas de 1 core,
# cambia cores=1 (mas lento pero mas estable dentro de notebooks en Windows).
N_CORES = min(2, os.cpu_count() or 1)

prevent_sleep()

with bayes_model:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        cores=N_CORES,
        return_inferencedata=True,
    )

allow_sleep()


## 9. Diagnóstico del modelo (MCMC)

- **`r_hat`**: debe ser muy cercano a 1.0 (idealmente ≤ 1.01). Valores más altos
  indican que las cadenas no convergieron al mismo posterior.
- **`ess_bulk` / `ess_tail`** (tamaño de muestra efectivo): cuanto más alto, más
  confiables son las estimaciones (idealmente > 400).

Con 180 variables (362 coeficientes en total entre `beta_draw` y `beta_away`), mostrar
la tabla completa de `az.summary` es poco legible. Por eso: primero se resumen los
hiperparámetros globales (interceptos y escalas del prior jerárquico), y luego se
muestran solo los 20 coeficientes de mayor magnitud de cada clase (de los 180 totales,
todos siguen dentro del `trace` guardado — esto es solo una cuestión de visualización,
no de modelado).

In [ ]:
# ============================================================
# 9. DIAGNOSTICO DEL MODELO
# ============================================================
summary_global = az.summary(trace, var_names=["intercept_draw", "intercept_away",
                                                "sigma_beta_draw", "sigma_beta_away"])
print("Resumen de hiperparametros globales:")
print(summary_global)

high_rhat = az.summary(trace, var_names=["beta_draw", "beta_away"])
high_rhat = high_rhat[high_rhat["r_hat"] > 1.01]
if len(high_rhat) > 0:
    print(f"\nAtencion: {len(high_rhat)} coeficientes con r_hat > 1.01 (posible falta de convergencia):")
    print(high_rhat.head(20))
else:
    print("\nTodos los coeficientes tienen r_hat <= 1.01: buena senal de convergencia.")


In [ ]:
# Trace plots de los hiperparametros globales (interceptos + escalas del prior jerarquico)
az.plot_trace(trace, var_names=["intercept_draw", "intercept_away", "sigma_beta_draw", "sigma_beta_away"])
plt.tight_layout()
plt.show()


In [ ]:
# Forest plots: top 20 coeficientes por magnitud (de los 180 totales por clase)
beta_draw_mean = trace.posterior["beta_draw"].mean(dim=["chain", "draw"]).values
beta_away_mean = trace.posterior["beta_away"].mean(dim=["chain", "draw"]).values

order_draw = np.argsort(-np.abs(beta_draw_mean))[:20]
order_away = np.argsort(-np.abs(beta_away_mean))[:20]

top20_features_draw = [all_feature_cols[i] for i in order_draw]
top20_features_away = [all_feature_cols[i] for i in order_away]

az.plot_forest(trace, var_names=["beta_draw"], coords={"feature": top20_features_draw},
                combined=True, figsize=(8, 10))
plt.title("Top 20 coeficientes beta_draw por magnitud (empate vs. gana local; de 180 totales)")
plt.tight_layout()
plt.show()

az.plot_forest(trace, var_names=["beta_away"], coords={"feature": top20_features_away},
                combined=True, figsize=(8, 10))
plt.title("Top 20 coeficientes beta_away por magnitud (gana visitante vs. gana local; de 180 totales)")
plt.tight_layout()
plt.show()


## 10. Evaluación en el conjunto de test

Igual que en Colab: reutilizamos el modelo ya entrenado, cambiamos los datos con
`pm.set_data` y generamos muestras predictivas posteriores.

In [ ]:
# ============================================================
# 10. EVALUACION EN EL CONJUNTO DE TEST
# ============================================================
with bayes_model:
    pm.set_data({"X_data": X_test_scaled, "y_data": np.zeros(len(y_test), dtype=int)})
    ppc_test = pm.sample_posterior_predictive(
        trace, var_names=["p", "y_obs"], random_seed=RANDOM_SEED
    )

# Probabilidad promedio del posterior para cada partido de test: shape (n_test, 3)
p_test_mean = ppc_test.posterior_predictive["p"].mean(dim=["chain", "draw"]).values
y_pred_test = p_test_mean.argmax(axis=1)

acc = accuracy_score(y_test, y_pred_test)
print(f"Accuracy en test: {acc:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred_test,
    target_names=["Gana local", "Empate", "Gana visitante"]
))

cm = confusion_matrix(y_test, y_pred_test)
cm_df = pd.DataFrame(
    cm,
    index=["Real: local", "Real: empate", "Real: visitante"],
    columns=["Pred: local", "Pred: empate", "Pred: visitante"],
)
print("\nMatriz de confusion:")
print(cm_df)

avg_prob_per_class = p_test_mean.mean(axis=0)
print("\nProbabilidad promedio (test set) por clase:")
for cls, name in zip(range(3), ["Gana local", "Empate", "Gana visitante"]):
    print(f"  {name}: {avg_prob_per_class[cls]:.4f}")


In [ ]:
# Matriz de confusion (grafica)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["Local", "Empate", "Visitante"])
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Local", "Empate", "Visitante"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Modelo Bayesiano MCMC (todas las variables)")
plt.tight_layout()
plt.show()


## 11. Guardar el modelo localmente (no en Google Drive)

Se crea una carpeta local `modelo_bayesiano_futbol/` (relativa a donde abriste
Jupyter) con dos archivos:

- **`trace_bayesian_mcmc.nc`**: el trace completo del MCMC (todas las cadenas y
  muestras posteriores), guardado con ArviZ en formato NetCDF.
- **`artifacts.pkl`**: un diccionario con todo lo necesario para volver a usar el
  modelo sin reentrenar — las 180 variables usadas (`feature_cols`), el `scaler`
  ajustado, el historial de equipos (`team_history`, necesario para proyectar
  partidos futuros), los nombres de clase, y los parámetros del feature engineering
  (`windows`, `base_metrics`, `std_metrics`).

In [ ]:
# ============================================================
# 11. GUARDAR MODELO LOCALMENTE
# ============================================================
MODEL_DIR = Path("modelo_bayesiano_futbol")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trace_path = MODEL_DIR / "trace_bayesian_mcmc.nc"
az.to_netcdf(trace, trace_path)
print("Trace guardado en:", trace_path.resolve())

artifacts = {
    "feature_cols": all_feature_cols,   # las 180 variables usadas por el modelo (todas, sin recorte)
    "top_features": all_feature_cols,   # se mantiene esta clave por compatibilidad: aqui es la lista completa
    "scaler_bayes": scaler,
    "team_history": long_df,            # historial largo por equipo, necesario para proyectar partidos futuros
    "class_names": ["Gana local", "Empate", "Gana visitante"],
    "windows": WINDOWS,
    "base_metrics": BASE_METRICS,
    "std_metrics": STD_METRICS,
}

artifacts_path = MODEL_DIR / "artifacts.pkl"
with open(artifacts_path, "wb") as f:
    pickle.dump(artifacts, f)

print("Artifacts guardados en:", artifacts_path.resolve())


## 12. Cargar el modelo guardado en futuras sesiones (sin reentrenar)

En una sesión nueva de Jupyter, después de correr la celda de imports (Sección 1),
cambia `LOAD_SAVED_MODEL = True` y ejecuta esta celda **en vez de** repetir las
Secciones 2 a 10. El `trace` no guarda el grafo del modelo, así que se reconstruye la
misma arquitectura con `build_bayesian_model(...)` (sin volver a muestrear) para poder
usar `pm.set_data` + `sample_posterior_predictive` sobre el `trace` cargado.

In [ ]:
# ============================================================
# 12. CARGAR MODELO GUARDADO (para usar en una sesion nueva sin reentrenar)
# ============================================================
LOAD_SAVED_MODEL = False  # cambia a True para cargar en vez de reentrenar

if LOAD_SAVED_MODEL:
    MODEL_DIR = Path("modelo_bayesiano_futbol")

    trace = az.from_netcdf(MODEL_DIR / "trace_bayesian_mcmc.nc")
    with open(MODEL_DIR / "artifacts.pkl", "rb") as f:
        artifacts = pickle.load(f)

    all_feature_cols = artifacts["feature_cols"]
    scaler = artifacts["scaler_bayes"]
    long_df = artifacts["team_history"]
    class_names = artifacts["class_names"]
    WINDOWS = artifacts["windows"]
    BASE_METRICS = artifacts["base_metrics"]
    STD_METRICS = artifacts["std_metrics"]

    # El objeto trace no incluye el grafo del modelo: se reconstruye la misma
    # arquitectura (mismos priors, mismas dims) para poder hacer posterior
    # predictive sampling con set_data. No se vuelve a muestrear (no pm.sample aqui).
    n_features = len(all_feature_cols)
    bayes_model = build_bayesian_model(
        X=np.zeros((1, n_features)), y=np.zeros(1, dtype=int), feature_names=all_feature_cols
    )
    print("Modelo y trace cargados desde:", MODEL_DIR.resolve())
else:
    print("LOAD_SAVED_MODEL=False: se sigue usando el modelo recien entrenado en esta sesion.")


## 13. Validación de nombres de equipos

Los nombres deben coincidir exactamente con los usados en el dataset (en inglés, p.ej.
`"France"`, no `"Francia"`). `validate_team_name` revisa si el nombre existe y, si no,
sugiere los nombres más parecidos usando `difflib` antes de fallar con un error claro
(mejor detenerse aquí que generar una predicción silenciosa con datos vacíos).

In [ ]:
# ============================================================
# 13. VALIDACION DE NOMBRES DE EQUIPOS
# ============================================================
known_teams = sorted(set(long_df["team"].unique()))
print(f"Equipos disponibles en el dataset: {len(known_teams)}")


def validate_team_name(team_name, known_teams=known_teams):
    """
    Verifica que team_name exista tal cual en el dataset. Si no existe, sugiere
    nombres parecidos (por ejemplo 'Francia' -> sugiere 'France') y lanza un
    error claro en vez de seguir con datos vacios/incorrectos.
    """
    if team_name in known_teams:
        return team_name

    suggestions = difflib.get_close_matches(team_name, known_teams, n=5, cutoff=0.6)
    msg = f"El equipo '{team_name}' no existe en el dataset."
    if suggestions:
        msg += f" Quisiste decir: {suggestions}?"
    else:
        msg += " No se encontraron nombres parecidos; revisa 'known_teams' para ver la lista completa."
    raise ValueError(msg)


# Ejemplo de validacion fallida (nombre en espanol en vez de ingles)
try:
    validate_team_name("Francia")
except ValueError as e:
    print("Ejemplo de aviso por nombre incorrecto:")
    print(" ", e)


## 14. Predicción de partidos futuros: `predict_future_match_bayesian`

`create_match_features` arma el mismo esquema de variables (`home_*`, `away_*`,
`diff_*`, las 180 completas) que usó el entrenamiento, tomando para cada equipo sus
**últimos** partidos disponibles en el histórico (incluyendo el más reciente, ya que
se proyecta un partido futuro que aún no se jugó), y escala con el **mismo** `scaler`
ajustado en la Sección 6.

`predict_future_match_bayesian(home_team, away_team, neutral=1)`:
1. Valida los nombres de los equipos.
2. Construye las features y las escala.
3. Usa el modelo (entrenado o cargado) para obtener probabilidades posteriores.
4. Muestra una tabla clara con las probabilidades (y su intervalo de credibilidad 94%).
5. Grafica una barra con las probabilidades.
6. Exporta un CSV con nombre automático, por ejemplo
   `prediction_France_vs_Paraguay_bayesian.csv`.

In [ ]:
# ============================================================
# 14. CREATE_MATCH_FEATURES + PREDICT_FUTURE_MATCH_BAYESIAN
# ============================================================
def get_latest_team_form(team_name, long_df, windows=WINDOWS):
    """
    Calcula las metricas de forma de un equipo usando sus ULTIMOS partidos
    disponibles en el historico (incluye el partido mas reciente, ya que se
    esta proyectando un partido futuro que todavia no se jugo).
    """
    hist = long_df[long_df["team"] == team_name].sort_values("date")
    stats = {}
    for w in windows:
        recent = hist.tail(w)
        for col in BASE_METRICS:
            stats[f"{col}_mean_{w}"] = recent[col].mean() if len(recent) > 0 else 0.0
        for col in STD_METRICS:
            stats[f"{col}_std_{w}"] = recent[col].std() if len(recent) > 1 else 0.0
    return stats


def create_match_features(home_team, away_team, neutral=1):
    """
    Construye el vector de variables (las 180 completas, ya escaladas) para un
    partido hipotetico entre home_team y away_team, usando el mismo esquema de
    features que el modelo de entrenamiento y el mismo scaler ya ajustado.
    """
    home_stats = get_latest_team_form(home_team, long_df)
    away_stats = get_latest_team_form(away_team, long_df)

    row = {}
    for col, val in home_stats.items():
        row[f"home_{col}"] = val
    for col, val in away_stats.items():
        row[f"away_{col}"] = val
    for col in home_stats:
        row[f"diff_{col}"] = home_stats[col] - away_stats[col]
    # 'neutral' se deja disponible por si se agrega como feature en una version futura
    row["_neutral"] = neutral

    match_df = pd.DataFrame([row])
    for c in all_feature_cols:
        if c not in match_df.columns:
            match_df[c] = 0.0
    match_df = match_df[all_feature_cols].fillna(0.0)

    match_scaled = scaler.transform(match_df.values)
    return match_scaled


def predict_future_match_bayesian(home_team, away_team, neutral=1, export_csv=True):
    """
    Predice el resultado de un partido futuro home_team vs away_team con el
    modelo Bayesiano MCMC ya entrenado/cargado. Muestra una tabla de
    probabilidades, una grafica de barras, y exporta un CSV automatico.
    """
    home_team = validate_team_name(home_team)
    away_team = validate_team_name(away_team)

    X_match_scaled = create_match_features(home_team, away_team, neutral=neutral)

    with bayes_model:
        pm.set_data({"X_data": X_match_scaled, "y_data": np.zeros(1, dtype=int)})
        ppc_match = pm.sample_posterior_predictive(trace, var_names=["p"], random_seed=RANDOM_SEED)

    # (chain, draw, 1, 3) -> (n_samples, 3)
    p_samples = ppc_match.posterior_predictive["p"].values.reshape(-1, 3)
    p_mean = p_samples.mean(axis=0)
    hdi_bounds = np.array([az.hdi(p_samples[:, k], hdi_prob=0.94) for k in range(3)])

    table = pd.DataFrame({
        "resultado": [f"{home_team} gana", "Empate", f"{away_team} gana"],
        "probabilidad": p_mean,
    })
    table["probabilidad_%"] = (table["probabilidad"] * 100).round(2)
    table["hdi_94_low_%"] = (hdi_bounds[:, 0] * 100).round(2)
    table["hdi_94_high_%"] = (hdi_bounds[:, 1] * 100).round(2)

    print(f"\nPrediccion Bayesiana MCMC: {home_team} vs {away_team}\n")
    print(table.to_string(index=False))

    plt.figure(figsize=(6, 4))
    colors = ["#1f77b4", "#7f7f7f", "#d62728"]
    plt.bar(table["resultado"], table["probabilidad_%"], color=colors)
    plt.ylabel("Probabilidad (%)")
    plt.title(f"{home_team} vs {away_team} (Modelo Bayesiano MCMC)")
    for i, v in enumerate(table["probabilidad_%"]):
        plt.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
    plt.ylim(0, max(table["probabilidad_%"]) + 15)
    plt.tight_layout()
    plt.show()

    if export_csv:
        safe_home = home_team.replace(" ", "_")
        safe_away = away_team.replace(" ", "_")
        csv_name = f"prediction_{safe_home}_vs_{safe_away}_bayesian.csv"
        table[["resultado", "probabilidad"]].to_csv(csv_name, index=False)
        print(f"\nArchivo exportado: {Path(csv_name).resolve()}")

    return table


## 15. Ejemplos de predicción

Predicción del partido objetivo (Francia vs Paraguay) y dos ejemplos adicionales para
mostrar que la función funciona con cualquier par de equipos del dataset.

In [ ]:
# Prediccion principal: Francia vs Paraguay
prediction_france_paraguay = predict_future_match_bayesian("France", "Paraguay", neutral=1)


In [ ]:
# Ejemplos adicionales
prediction_mexico_england = predict_future_match_bayesian("Mexico", "England", neutral=1)


In [ ]:
prediction_brazil_norway = predict_future_match_bayesian("Brazil", "Norway", neutral=1)


## 16. Qué cambió respecto a la versión de Colab

| Aspecto | Versión Colab | Versión local (este notebook) |
|---|---|---|
| **Rutas** | Hardcodeadas a `/content/...` | `REPO_PATH` y `DATA_PATH` configurables, con `pathlib.Path` (compatible con Windows, Mac, Linux) |
| **Google Drive** | No se usaba `drive.mount()` en el original, pero tampoco había persistencia entre sesiones | No aplica Drive; en su lugar hay una carpeta local `modelo_bayesiano_futbol/` con persistencia real en disco |
| **Instalación de librerías** | `!pip install` dentro de una celda (magic command de Colab) | Se instalan una sola vez al crear el entorno de conda (Sección 0); ya no hay `!pip install` en el notebook |
| **Clonado del repo** | `git clone` a una ruta fija de `/content/` | `git clone` opcional a `REPO_PATH` configurable, con alternativa manual (descargar el ZIP y apuntar `DATA_PATH`/`REPO_PATH` a la carpeta local) |
| **Selección de variables** | Top-40 por importancia de Random Forest, para que el MCMC terminara dentro del tiempo de Colab | **Todas las 180 variables**, con un prior jerárquico de regularización en `beta_draw`/`beta_away` para manejar la alta dimensionalidad sin descartar variables |
| **Persistencia del modelo** | Ninguna (se perdía todo al desconectar Colab) | `trace_bayesian_mcmc.nc` (ArviZ/NetCDF) + `artifacts.pkl` (pickle) en `modelo_bayesiano_futbol/`, con una celda para recargarlos en sesiones futuras sin reentrenar |
| **Ejecución** | Notebook en la nube, con límite de tiempo de sesión y desconexiones | Jupyter Lab local, sin límite de tiempo; se agregó una sección para evitar que Windows suspenda el equipo durante el entrenamiento |
| **Validación de equipos** | No existía | `validate_team_name()` con sugerencias vía `difflib` si el nombre no coincide exactamente (ej. "Francia" -> sugiere "France") |
| **Dependencias** | Instaladas por Colab en cada sesión efímera | Fijadas en un entorno de conda reproducible (`futbol-bayes`), con kernel de Jupyter registrado explícitamente |

**Lo que NO cambió:** el pipeline completo (limpieza, feature engineering sin leakage,
split temporal 80/20, arquitectura del modelo bayesiano — softmax multinomial con clase
base "gana local", `pm.Categorical`, configuración de `pm.sample` con
`draws=1000, tune=1000, chains=2, target_accept=0.9, random_seed=42`), la evaluación en
test, y la lógica de predicción para partidos futuros.
